# Lung Cancer Prediction using Sagemaker Training Pipeline
# Used the prepared file from S3 featues/ folder -> data_preparation_handoff.ipynb

# Imports & Setup

In [87]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import os
import io
import time
from sagemaker.xgboost import XGBoost
from sklearn.utils import resample
from sagemaker.inputs import TrainingInput
from sagemaker import image_uris, get_execution_role
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, confusion_matrix
)
 
session   = sagemaker.Session()
role      = get_execution_role()
bucket    = "prognostica-cancer-project"
prefix    = "lung-cancer"
s3_client = boto3.client("s3")
 
print(f"Role   : {role}")
print(f"Bucket : {bucket}")
print(f"Region : {session.boto_region_name}")

Role   : arn:aws:iam::381491860224:role/LabRole
Bucket : prognostica-cancer-project
Region : us-east-1


In [88]:
# Environmanetal setup to retrieve execution role, define S3 bucket and prefix for storing project files and import thre required libraries.

# Training Preprocessing

In [89]:
train_df = pd.read_csv("s3://prognostica-cancer-project/features/lung_train.csv")

# Encode gender
train_df["gender"] = train_df["gender"].map({"M": 1, "F": 0})

# Encode target
train_df["lung_cancer"] = (
    train_df["lung_cancer"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({"YES": 1, "NO": 0})
)

# Convert boolean age groups to integers
bool_cols = [
    "age_group_41-50",
    "age_group_51-60",
    "age_group_61-70",
    "age_group_71+"
]

for col in bool_cols:
    train_df[col] = train_df[col].astype(int)

In [90]:
# Loading the CSV training data and encoding categorical features to boolean.

# Balance the Training Data

In [91]:
df_yes = train_df[train_df.lung_cancer == 1]
df_no = train_df[train_df.lung_cancer == 0]

df_no_upsampled = resample(
    df_no,
    replace=True,
    n_samples=len(df_yes),
    random_state=42
)

balanced_df = pd.concat([df_yes, df_no_upsampled]).sample(frac=1, random_state=42)

In [92]:
# Balancing the dataset using split data, unsampling the minority class and combine and shuffle to create the balanced training dataset.

# Save and Upload the Balanced Training File

In [93]:
balanced_df.to_csv("lung_train_balanced.csv", index=False)

s3_client.upload_file(
    "lung_train_balanced.csv",
    bucket,
    "features/lung_train_balanced.csv"
)

In [94]:
# Saved trianing data and uploaded to S3 bucket

# Validation Preprocessing

In [95]:
val_df = pd.read_csv("s3://prognostica-cancer-project/features/lung_val.csv")

val_df["gender"] = val_df["gender"].map({"M": 1, "F": 0})

val_df["lung_cancer"] = (
    val_df["lung_cancer"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({"YES": 1, "NO": 0})
)

for col in bool_cols:
    val_df[col] = val_df[col].astype(int)

val_df.to_csv("lung_val_processed.csv", index=False)

s3_client.upload_file(
    "lung_val_processed.csv",
    bucket,
    "features/lung_val_processed.csv"
)

In [96]:
# Prepared validation data and converted boolean columns (target and gender) to integers. 

# Define Train and Validation Paths

In [97]:
train_path = "s3://prognostica-cancer-project/features/lung_train_balanced.csv"
val_path = "s3://prognostica-cancer-project/features/lung_val_processed.csv"


In [98]:
# Create the Training Inputs Objects

In [99]:
train_input = TrainingInput(
    s3_data=train_path,
    content_type="text/csv"
)

val_input = TrainingInput(
    s3_data=val_path,
    content_type="text/csv"
)

# Train the Model

In [100]:
from sagemaker.estimator import Estimator
from sagemaker import image_uris

container = image_uris.retrieve("xgboost", session.boto_region_name, version="1.5-1")

xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    hyperparameters={
        "objective": "binary:logistic",
        "num_round": 200
    },
    sagemaker_session=session
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [101]:
xgb.fit({"train": train_input, "validation": val_input})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-04-13-19-41-25-583


2026-04-13 19:41:30 Starting - Starting the training job...
2026-04-13 19:41:45 Starting - Preparing the instances for training...
2026-04-13 19:42:07 Downloading - Downloading input data...
2026-04-13 19:42:53 Downloading - Downloading the training image......
2026-04-13 19:43:54 Training - Training image download completed. Training in progress../miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-04-13 19:43:55.742 ip-10-0-103-53.ec2.internal:8 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-04-13 19:43:55.764 ip-10-0-103-53.ec2.internal:8 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-04-13:19:43:56:INFO] Imported framework sagemaker_xgboost_container.training
[2026-04-13:19:43:56:INFO] Failed to parse hyperparameter objective value binary

In [102]:
# Ran training job using XGBoost estimator. Evaluated model performance using acuracy, ROC-AUC, confusion matrix and classification report.

# Deploy the Model

In [103]:
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer

# Create a SageMaker Model Object with the trianed model artifact stored in S3
model = Model(
    image_uri=container,
    model_data="s3://prognostica-cancer-project/output/sagemaker-xgboost-2026-04-10-23-00-50-177/output/model.tar.gz",
    role=role,
    sagemaker_session=session
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    wait=True
)

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-04-13-19-44-43-214
INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2026-04-13-19-44-43-879
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2026-04-13-19-44-43-879


------!

# Attach Serializer

In [104]:
predictor = Predictor(
    endpoint_name="sagemaker-xgboost-2026-04-10-23-31-41-495",
    sagemaker_session=session,
    serializer=CSVSerializer()
)

In [105]:
# Attach CSVSerializer so the endpoint receives input in the correct CSV format.
# Required for reproducibility because the model expects CSV-formatted payloads.

# Predictions

In [106]:
X_test = val_df.drop("lung_cancer", axis=1)
y_test = val_df["lung_cancer"]

In [107]:
payload = X_test.to_csv(index=False, header=False)

# Get raw predictions from the endpoint
raw_preds = predictor.predict(payload)

#  Decode bytes → string
raw_preds = raw_preds.decode("utf-8")

# Convert predictions to floats
preds = np.array([
    float(x) for x in raw_preds.split("\n")
    if x.strip() != ""
])


In [108]:
# Defined X_test and y_test before generating predictions.
# This workflow used pre-split datasets (train and validation), so X_test is not created automatically.
# Here we used the validation set as the test set for endpoint evaluation.

# Convert Predictions

In [109]:
preds = np.array([float(x) for x in raw_preds.split("\n") if x.strip() != ""])


In [110]:
# Convert raw string outputs into float probabilities for evaluation.


# Evaluate

In [111]:
print("Accuracy:", accuracy_score(y_test, preds > 0.5))
print("AUC:", roc_auc_score(y_test, preds))
print(confusion_matrix(y_test, preds > 0.5))


Accuracy: 0.521117166212534
AUC: 0.4883016588296761
[[ 181  203]
 [1203 1349]]


In [112]:
# Compute accuracy, AUC, and confusion matrix to assess model performance.
# These metrics confirm whether the deployed model produces meaningful predictions.

In [113]:
# Although the workflow performed correctly end‑to‑end, the model showed low accuracy and AUC; this was due to the limited predictive value of the available datasets, but the pipeline is ready to be used with clinical data (higher-quaility). 